In [1]:
import timm
import torch
import torch.nn as nn

import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from dataset import JetbotDataset

In [2]:
from torch.utils.data import WeightedRandomSampler

def make_balanced_sampler(dataset, straight_threshold=0.05):
    """
    Returns a WeightedRandomSampler that balances:
      - straight  (|turn| <= threshold)
      - turn left (turn  <  -threshold)
      - turn right(turn  >   threshold)
    """
    turns = np.array([dataset[i][1][1] for i in range(len(dataset))])  # all turn values

    left_mask     = turns <  -straight_threshold
    right_mask    = turns >   straight_threshold
    straight_mask = np.abs(turns) <= straight_threshold

    n_left     = left_mask.sum()
    n_right    = right_mask.sum()
    n_straight = straight_mask.sum()

    print(f"Dataset composition — straight: {n_straight}, left: {n_left}, right: {n_right}")

    # Each class gets weight = 1 / class_size  → equal expected frequency
    w_left     = 1.0 / n_left     if n_left     > 0 else 0.0
    w_right    = 1.0 / n_right    if n_right    > 0 else 0.0
    w_straight = 1.0 / n_straight if n_straight > 0 else 0.0

    sample_weights = np.zeros(len(dataset))
    sample_weights[left_mask]     = w_left
    sample_weights[right_mask]    = w_right
    sample_weights[straight_mask] = w_straight

    return WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(dataset),
        replacement = True,
    )

In [3]:
# ==========================================
# PATHS
# ==========================================

csv_path = "../../data/dataset_retarded.csv"
img_path = "../../data/dataset_images"

model_name = 'timm/mobilenetv4_conv_small_050.e3000_r224_in1k'

# ==========================================
# LOAD CSV
# ==========================================

df = pd.read_csv(csv_path, dtype={"frame_timestamp": str})

print(df.head())
print(f"\nDataset size: {len(df)}")

# ==========================================
# MODEL & TRANSFORMS
# ==========================================

model = timm.create_model(model_name, pretrained=True)

# get model specific transforms (normalization, resize)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# ==========================================
# FULL DATASET
# ==========================================

dataset = JetbotDataset(
    dataframe=df,
    image_dir=img_path,
    transform=transform
)

# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(17)
)

sampler = make_balanced_sampler(train_dataset)

print(f"Train size: {len(train_dataset)}")
print(f"Test size : {len(test_dataset)}")

# ==========================================
# DATALOADERS
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler = sampler,
    # shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# ==========================================
# SANITY CHECK
# ==========================================

images, targets = next(iter(train_loader))

print("Images shape :", images.shape)
print("Targets shape:", targets.shape)

            frame_timestamp  left_motor  right_motor  throttle      turn  \
0         1778600883.155138    0.128237     0.125669  0.427451  0.000000   
1  1778600883.155138_mirror    0.125669     0.128237  0.427451 -0.000000   
2         1778600883.277637    0.090590     0.088775  0.301961  0.000000   
3  1778600883.277637_mirror    0.088775     0.090590  0.301961 -0.000000   
4         1778600883.422115    0.065647     0.113219  0.301961 -0.207843   

   left_trigger  right_trigger          control_timestamp  
0           0.0            0.0         1778600883.3282287  
1           0.0            0.0  1778600883.3282287_mirror  
2           0.0            0.0         1778600883.4872782  
3           0.0            0.0  1778600883.4872782_mirror  
4           0.0            0.0          1778600883.620211  

Dataset size: 3896


  0%|          | 0/3896 [00:00<?, ?it/s]

Dataset composition — straight: 2231, left: 430, right: 455
Train size: 3116
Test size : 780
Images shape : torch.Size([32, 3, 224, 224])
Targets shape: torch.Size([32, 2])


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
model = timm.create_model(model_name, pretrained=True)

In [6]:
transform

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
)

In [7]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    steering_loss_sum = 0.0
    mae_sum = 0.0

    dir_sum, dir_count       = 0, 0   # directional acc  (|target| > 0.05)
    zero_correct, zero_count = 0, 0   # zero acc          (|target| <= 0.05)

    for images, targets in loader:
        images  = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = (
            nn.functional.mse_loss(outputs[:, 0], targets[:, 0]) +
            2.0 * nn.functional.mse_loss(outputs[:, 1], targets[:, 1])
        )
        loss.backward()
        optimizer.step()

        total_loss     += loss.item()
        mae_sum        += torch.abs(outputs[:, 1] - targets[:, 1]).sum().item()
        steering_loss_sum += nn.functional.l1_loss(outputs[:, 1], targets[:, 1]).item()

        pred_turn   = outputs[:, 1]
        target_turn = targets[:, 1]

        # --- non-zero steering: sign must match ---
        dir_mask = target_turn.abs() > 0.05
        if dir_mask.sum() > 0:
            dir_sum   += (torch.sign(pred_turn[dir_mask]) == torch.sign(target_turn[dir_mask])).float().sum().item()
            dir_count += dir_mask.sum().item()

        # --- zero steering: prediction should also be near zero ---
        zero_mask = target_turn.abs() <= 0.05
        if zero_mask.sum() > 0:
            zero_correct += (pred_turn[zero_mask].abs() <= 0.05).float().sum().item()
            zero_count   += zero_mask.sum().item()

    n = len(loader.dataset)
    return (
        total_loss        / len(loader),
        steering_loss_sum / len(loader),
        mae_sum           / n,
        dir_sum           / max(dir_count, 1),   # acc on turns
        zero_correct      / max(zero_count, 1)  # acc on straights
    )

In [8]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    steering_loss_sum = 0.0
    mae_sum = 0.0

    dir_sum, dir_count       = 0, 0
    zero_correct, zero_count = 0, 0

    for images, targets in loader:
        images  = images.to(device)
        targets = targets.to(device)
        outputs = model(images)

        loss = (
            nn.functional.mse_loss(outputs[:, 0], targets[:, 0]) +
            2.0 * nn.functional.mse_loss(outputs[:, 1], targets[:, 1])
        )
        total_loss        += loss.item()
        mae_sum           += torch.abs(outputs[:, 1] - targets[:, 1]).sum().item()
        steering_loss_sum += nn.functional.l1_loss(outputs[:, 1], targets[:, 1]).item()

        pred_turn   = outputs[:, 1]
        target_turn = targets[:, 1]

        dir_mask = target_turn.abs() > 0.05
        if dir_mask.sum() > 0:
            dir_sum   += (torch.sign(pred_turn[dir_mask]) == torch.sign(target_turn[dir_mask])).float().sum().item()
            dir_count += dir_mask.sum().item()

        zero_mask = target_turn.abs() <= 0.05
        if zero_mask.sum() > 0:
            zero_correct += (pred_turn[zero_mask].abs() <= 0.05).float().sum().item()
            zero_count   += zero_mask.sum().item()

    n = len(loader.dataset)
    return (
        total_loss        / len(loader),
        steering_loss_sum / len(loader),
        mae_sum           / n,
        dir_sum           / max(dir_count, 1),   # acc on turns
        zero_correct      / max(zero_count, 1)  # acc on straights
    )

In [9]:
model = timm.create_model(model_name, pretrained=True)

model.classifier = nn.Sequential(
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 2),
    nn.Tanh()
)
model = model.to(device)

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

In [10]:
# model

In [11]:
# 1. Unfreeze last 2 blocks of the backbone
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last blocks (adjust layer names via print(model) to confirm)
for param in model.blocks[-3:].parameters():
    param.requires_grad = True

for param in model.conv_head.parameters():
    param.requires_grad = True
    
for param in model.norm_head.parameters():
    param.requires_grad = True
    
for param in model.classifier.parameters():
    param.requires_grad = True

# 2. Layer-wise LR: backbone gets 10x less than head
optimizer = torch.optim.Adam([
    {"params": model.blocks[-3:].parameters(), "lr": 1e-4},
    {"params": model.conv_head.parameters(), "lr": 1e-3},
    {"params": model.norm_head.parameters(), "lr": 1e-3},
    {"params": model.classifier.parameters(),  "lr": 1e-3},
])

In [12]:
num_epochs = 50
os.makedirs("./checkpoints", exist_ok=True)
for epoch in tqdm(range(num_epochs)):

    train_loss, train_steer_loss, train_mae, train_dir, train_zero_acc = train_one_epoch(model, train_loader)
    val_loss, val_steer_loss, val_mae, val_dir, val_zero_acc = evaluate(model, test_loader)

    torch.save({
        "model_state_dict": model.state_dict()
    }, f"./checkpoints/checkpoints_{epoch}")

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train loss: {train_loss:.4f} | MAE: {train_mae:.4f} | STEER: {train_steer_loss:.4f} | DIR: {train_dir:.4f} | ZERO: {train_zero_acc:.4f}")
    print(f"Val loss  : {val_loss:.4f} | MAE: {val_mae:.4f} | STEER: {val_steer_loss:.4f} | DIR: {val_dir:.4f} | ZERO: {val_zero_acc:.4f}")
    print("-" * 40)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50
Train loss: 0.5648 | MAE: 0.3618 | STEER: 0.3620 | DIR: 0.8458 | ZERO: 0.1409
Val loss  : 0.3682 | MAE: 0.2879 | STEER: 0.2947 | DIR: 0.8959 | ZERO: 0.1342
----------------------------------------
Epoch 2/50
Train loss: 0.3037 | MAE: 0.2585 | STEER: 0.2590 | DIR: 0.9428 | ZERO: 0.2031
Val loss  : 0.3310 | MAE: 0.2622 | STEER: 0.2665 | DIR: 0.9005 | ZERO: 0.2111
----------------------------------------
Epoch 3/50
Train loss: 0.2648 | MAE: 0.2355 | STEER: 0.2361 | DIR: 0.9581 | ZERO: 0.2179
Val loss  : 0.3252 | MAE: 0.2597 | STEER: 0.2641 | DIR: 0.8914 | ZERO: 0.2165
----------------------------------------
Epoch 4/50
Train loss: 0.2143 | MAE: 0.2151 | STEER: 0.2151 | DIR: 0.9610 | ZERO: 0.2257
Val loss  : 0.3302 | MAE: 0.2545 | STEER: 0.2590 | DIR: 0.8914 | ZERO: 0.3131
----------------------------------------
Epoch 5/50
Train loss: 0.1971 | MAE: 0.1949 | STEER: 0.1950 | DIR: 0.9618 | ZERO: 0.2705
Val loss  : 0.3316 | MAE: 0.2564 | STEER: 0.2574 | DIR: 0.8643 | ZERO: 0.2343
-

In [13]:
checkpoint = torch.load("./checkpoints/checkpoints_48", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [14]:
torch.save({
    "model_state_dict": model.state_dict()
}, "../models/mobilenet_tiny_retarded_over_sampling")

In [16]:
checkpoint = torch.load("../models/mobilenet_tiny2", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>